In [9]:
# Instalamos las librerias externas que necesitamos para generar PDFs y codigos QR
!pip install fpdf2 qrcode[pil] pillow -q

In [10]:
# Importamos las librerias que vamos a usar en todo el sistema
import hashlib   # Para generar los codigos hash SHA-256 de cada bloque
import json      # Para convertir los datos del bloque a texto antes de hashearlos
import time      # Para registrar la fecha y hora exacta en cada bloque
import qrcode    # Para generar los codigos QR de cada certificado
from fpdf import FPDF  # Para crear los certificados en formato PDF

# Definimos la clase que representa un bloque individual de la cadena
class Block:

     # Este metodo se ejecuta automaticamente al crear un nuevo bloque
    def __init__(self, index, timestamp, data, previous_hash):
        self.index = index              # Numero de orden del bloque en la cadena
        self.timestamp = timestamp      # Momento exacto en que se creo el bloque
        self.data = data                # Informacion que guarda el bloque (el certificado)
        self.previous_hash = previous_hash  # Hash del bloque anterior para encadenarlos
        self.nonce = 0                  # Numero que se ajusta durante el Proof of Work
        self.hash = self.compute_hash() # Calculamos el hash del bloque al crearlo

    # Calculamos el hash unico del bloque usando todos sus datos
    def compute_hash(self):
        # Convertimos todos los datos del bloque a un texto ordenado
        block_string = json.dumps({
            "index": self.index,
            "timestamp": self.timestamp,
            "data": self.data,
            "previous_hash": self.previous_hash,
            "nonce": self.nonce
        }, sort_keys=True)
        # Aplicamos SHA-256 al texto y devolvemos el resultado en hexadecimal
        return hashlib.sha256(block_string.encode()).hexdigest()

print("✅ Block listo")

✅ Block listo


In [11]:
# Definimos la clase que maneja toda la cadena de bloques
class Blockchain:

    # Al crear la cadena configuramos la dificultad y generamos el primer bloque
    def __init__(self, difficulty=2):
        self.chain = []             # Lista donde guardamos todos los bloques
        self.difficulty = difficulty  # Cantidad de ceros que debe empezar el hash (Proof of Work)
        self.create_genesis_block()

    # El bloque genesis es el primer bloque de la cadena, no tiene datos de certificado
    def create_genesis_block(self):
        genesis = Block(0, time.time(), "Bloque Genesis - MINEDUC Guatemala", "0")
        genesis.hash = self.proof_of_work(genesis)
        self.chain.append(genesis)

    # Propiedad para acceder facilmente al ultimo bloque de la cadena
    @property
    def last_block(self):
        return self.chain[-1]

    # Buscamos un nonce que haga que el hash empiece con la cantidad de ceros requerida
    def proof_of_work(self, block):
        block.nonce = 0
        computed = block.compute_hash()
        # Seguimos incrementando el nonce hasta cumplir la condicion de dificultad
        while not computed.startswith("0" * self.difficulty):
            block.nonce += 1
            computed = block.compute_hash()
        return computed

    # Agregamos un nuevo bloque con datos a la cadena
    def add_block(self, data):
        previous_hash = self.last_block.hash  # Tomamos el hash del ultimo bloque
        new_block = Block(
            index=len(self.chain),
            timestamp=time.time(),
            data=data,
            previous_hash=previous_hash
        )
        new_block.hash = self.proof_of_work(new_block)  # Aplicamos Proof of Work
        self.chain.append(new_block)
        return new_block  # Retornamos el bloque para capturar su numero

    # Verificamos que todos los bloques de la cadena sean validos e integros
    def is_chain_valid(self):
        for i in range(1, len(self.chain)):
            current  = self.chain[i]
            previous = self.chain[i - 1]
            # Comprobamos que el hash almacenado coincide con el hash calculado
            if current.hash != current.compute_hash():
                print(f"  [FALLO] Bloque {i}: hash no coincide")
                return False
            # Comprobamos que el enlace con el bloque anterior es correcto
            if current.previous_hash != previous.hash:
                print(f"  [FALLO] Bloque {i}: enlace roto")
                return False
        print("  [OK] La cadena es valida")
        return True

    # Buscamos un certificado especifico por su hash de verificacion
    def buscar_por_hash(self, hash_buscado):
        for block in self.chain:
            # Solo revisamos bloques que contengan datos de certificados
            if isinstance(block.data, dict):
                if block.data.get("hash_cert") == hash_buscado.upper():
                    return block.data
        return None  # Si no encontramos el hash, devolvemos None

# Creamos la cadena principal del sistema
cadena = Blockchain(difficulty=2)
print("✅ Blockchain de MINEDUC iniciada")

✅ Blockchain de MINEDUC iniciada


In [12]:
# Funcion que registra un nuevo certificado de ingles en la blockchain
def emitir_certificado(nombre, colegio, nivel_ingles, anio_graduacion):

    # Obtenemos la fecha y hora actual del momento de emision
    fecha = time.strftime("%d/%m/%Y %H:%M:%S")

    # Generamos un hash unico de 12 caracteres combinando los datos del estudiante
    contenido = f"{nombre}|{colegio}|{nivel_ingles}|{anio_graduacion}"
    hash_cert = hashlib.sha256(contenido.encode()).hexdigest()[:12].upper()

    # Organizamos todos los datos del certificado en un diccionario
    datos = {
        "tipo":            "CERT_INGLES_MINEDUC",  # Tipo de registro en la cadena
        "nombre":          nombre,                  # Nombre completo del estudiante
        "colegio":         colegio,                 # Establecimiento educativo
        "nivel_ingles":    nivel_ingles,            # Nivel segun marco CEFR (A1 a C2)
        "anio_graduacion": str(anio_graduacion),    # Año en que se graduo
        "fecha_emision":   fecha,                   # Fecha en que se emite el certificado
        "institucion":     "Ministerio de Educacion de Guatemala - MINEDUC",
        "hash_cert":       hash_cert                # Codigo unico de verificacion
    }

    # Agregamos el certificado como un nuevo bloque en la cadena
    bloque = cadena.add_block(datos)
    print(f"✅ Certificado emitido | Bloque #{bloque.index} | Hash: {hash_cert} | {nombre}")
    return hash_cert  # Devolvemos el hash para poder generar el PDF

print("✅ Funcion emitir_certificado lista")

✅ Funcion emitir_certificado lista


In [13]:
print("=" * 60)
print("   MINEDUC - Emision de Certificados de Ingles 2024")
print("=" * 60)

# Emitimos un certificado por cada estudiante graduado
# Cada llamada registra un bloque nuevo en la cadena
h1 = emitir_certificado("Andrea Morales Cifuentes", "Colegio Americano de Guatemala",          "B2 - Intermedio Alto", 2024)
h2 = emitir_certificado("Jose Pablo Herrera Ruiz",  "Instituto Nacional Central para Varones", "A2 - Basico",          2024)
h3 = emitir_certificado("Sofia Ramirez Castillo",   "Colegio Monte Maria, Guatemala",          "C1 - Avanzado",        2024)
h4 = emitir_certificado("Diego Flores Monterroso",  "Liceo Guatemala, Zona 1",                 "B1 - Intermedio",      2024)
h5 = emitir_certificado("Valentina Estrada Lopez",  "Colegio Salesiano Don Bosco",             "A1 - Principiante",    2024)

# Guardamos todos los hashes en una lista para usarlos despues
hashes = [h1, h2, h3, h4, h5]

print()
# Verificamos que la cadena no ha sido alterada
print("Validando integridad de la cadena:")
cadena.is_chain_valid()
print(f"Total de bloques: {len(cadena.chain)} (1 genesis + 5 certificados)")

   MINEDUC - Emision de Certificados de Ingles 2024
✅ Certificado emitido | Bloque #1 | Hash: FF43F455D2D9 | Andrea Morales Cifuentes
✅ Certificado emitido | Bloque #2 | Hash: 0222DDA12BCD | Jose Pablo Herrera Ruiz
✅ Certificado emitido | Bloque #3 | Hash: A29E75E31E7A | Sofia Ramirez Castillo
✅ Certificado emitido | Bloque #4 | Hash: 2A52CF21CDFD | Diego Flores Monterroso
✅ Certificado emitido | Bloque #5 | Hash: 68B40421EE60 | Valentina Estrada Lopez

Validando integridad de la cadena:
  [OK] La cadena es valida
Total de bloques: 6 (1 genesis + 5 certificados)


In [14]:
# Funcion que permite verificar si un certificado es autentico
def verificar(hash_buscado):
    print("=" * 45)
    print(f"Buscando hash: {hash_buscado.upper()}")
    print("=" * 45)

    # Buscamos el hash en todos los bloques de la cadena
    datos = cadena.buscar_por_hash(hash_buscado)

    if datos:
        # Si lo encontramos, el certificado es autentico
        print("✅  CERTIFICADO AUTENTICO")
        print("-" * 45)
        print(f"  Nombre:             {datos['nombre']}")
        print(f"  Colegio:            {datos['colegio']}")
        print(f"  Nivel de ingles:    {datos['nivel_ingles']}")
        print(f"  Año de graduacion:  {datos['anio_graduacion']}")
        print(f"  Fecha emision:      {datos['fecha_emision']}")
        print(f"  Institucion:        {datos['institucion']}")
        print(f"  Hash:               {datos['hash_cert']}")
    else:
        # Si no lo encontramos, el certificado no existe o es falso
        print("❌  CERTIFICADO NO ENCONTRADO")
        print("    Este hash no existe en la blockchain.")
        print("    El documento puede ser FALSO.")

# Prueba 1: verificamos un certificado real
print("--- PRUEBA 1: hash correcto ---")
verificar(h1)
print()
# Prueba 2: intentamos verificar un hash inventado
print("--- PRUEBA 2: hash falso ---")
verificar("HASH000FALSO")

--- PRUEBA 1: hash correcto ---
Buscando hash: FF43F455D2D9
✅  CERTIFICADO AUTENTICO
---------------------------------------------
  Nombre:             Andrea Morales Cifuentes
  Colegio:            Colegio Americano de Guatemala
  Nivel de ingles:    B2 - Intermedio Alto
  Año de graduacion:  2024
  Fecha emision:      04/05/2026 00:47:07
  Institucion:        Ministerio de Educacion de Guatemala - MINEDUC
  Hash:               FF43F455D2D9

--- PRUEBA 2: hash falso ---
Buscando hash: HASH000FALSO
❌  CERTIFICADO NO ENCONTRADO
    Este hash no existe en la blockchain.
    El documento puede ser FALSO.


In [15]:
# Funcion que genera el certificado visual en formato PDF con codigo QR
def generar_pdf(hash_cert):

    # Buscamos los datos del certificado en la cadena usando el hash
    datos = cadena.buscar_por_hash(hash_cert)
    if not datos:
        print(f"❌ Hash no encontrado: {hash_cert}")
        return

    # Generamos el codigo QR que contiene el hash de verificacion
    qr = qrcode.make(f"MINEDUC Guatemala | Hash: {hash_cert}")
    qr_path = f"/tmp/qr_{hash_cert}.png"
    qr.save(qr_path)

    # Creamos el documento PDF
    pdf = FPDF()
    pdf.set_auto_page_break(auto=False)  # Desactivamos el salto de pagina automatico
    pdf.add_page()

    # ── Encabezado con color institucional MINEDUC ──
    pdf.set_fill_color(24, 95, 165)
    pdf.rect(0, 0, 210, 36, "F")
    pdf.set_text_color(255, 255, 255)
    pdf.set_font("Helvetica", "B", 15)
    pdf.set_xy(0, 6)
    pdf.cell(210, 9, "MINISTERIO DE EDUCACION DE GUATEMALA", align="C", ln=True)
    pdf.set_font("Helvetica", "", 10)
    pdf.set_x(0)
    pdf.cell(210, 7, "Certificado Oficial de Nivel de Ingles", align="C", ln=True)
    pdf.set_x(0)
    pdf.cell(210, 7, "Educacion Diversificada  |  Registro en Blockchain", align="C", ln=True)

    # ── Datos del estudiante ──
    pdf.set_text_color(30, 30, 30)
    pdf.set_xy(0, 46)
    pdf.set_font("Helvetica", "", 12)
    pdf.cell(210, 9, "Se certifica que el(la) estudiante:", align="C", ln=True)

    # Nombre del estudiante en grande y azul
    pdf.set_font("Helvetica", "B", 19)
    pdf.set_text_color(24, 95, 165)
    pdf.set_x(0)
    pdf.cell(210, 12, datos["nombre"].upper(), align="C", ln=True)

    # Colegio y nivel de ingles
    pdf.set_font("Helvetica", "", 12)
    pdf.set_text_color(30, 30, 30)
    pdf.set_x(0)
    pdf.cell(210, 8, f"del establecimiento: {datos['colegio']}", align="C", ln=True)
    pdf.set_x(0)
    pdf.cell(210, 8, "ha alcanzado el siguiente nivel de dominio del idioma ingles:", align="C", ln=True)

    # Nivel en grande y azul
    pdf.set_font("Helvetica", "B", 17)
    pdf.set_text_color(24, 95, 165)
    pdf.set_x(0)
    pdf.cell(210, 11, datos["nivel_ingles"], align="C", ln=True)

    # Año de graduacion y fecha de emision
    pdf.set_font("Helvetica", "", 11)
    pdf.set_text_color(30, 30, 30)
    pdf.set_x(0)
    pdf.cell(210, 9, f"Año de graduacion: {datos['anio_graduacion']}     |     Fecha de emision: {datos['fecha_emision']}", align="C", ln=True)

    # ── Linea separadora ──
    y_sep = pdf.get_y() + 4
    pdf.set_draw_color(200, 200, 200)
    pdf.line(20, y_sep, 190, y_sep)

    # ── Hash y texto del QR ──
    pdf.set_xy(0, y_sep + 5)
    pdf.set_font("Courier", "", 9)
    pdf.set_text_color(130, 130, 130)
    pdf.cell(210, 6, f"Hash de verificacion blockchain: {hash_cert}", align="C", ln=True)
    pdf.set_font("Helvetica", "", 9)
    pdf.set_x(0)
    pdf.cell(210, 6, "Escanea el codigo QR para verificar la autenticidad de este certificado", align="C", ln=True)

    # ── Imagen del QR centrada ──
    qr_y = pdf.get_y() + 4
    pdf.image(qr_path, x=80, y=qr_y, w=50)

    # ── Pie de pagina fijo al fondo del documento ──
    pdf.set_fill_color(24, 95, 165)
    pdf.rect(0, 272, 210, 25, "F")
    pdf.set_text_color(255, 255, 255)
    pdf.set_xy(0, 279)
    pdf.set_font("Helvetica", "", 8)
    pdf.cell(210, 6, "MINEDUC Guatemala | Documento valido solo con verificacion blockchain", align="C")

    # ── Guardamos el PDF en Colab ──
    ruta = f"/content/{hash_cert}_certificado.pdf"
    pdf.output(ruta)
    print(f"✅ PDF generado: {hash_cert}_certificado.pdf")

# Generamos el PDF para cada uno de los 5 certificados
print("Generando PDFs...\n")
for h in hashes:
    generar_pdf(h)

print("\n✅ Todos los PDFs generados.")
print("👉 Panel izquierdo → icono 📁 → clic derecho sobre cada PDF → Descargar")

Generando PDFs...

✅ PDF generado: FF43F455D2D9_certificado.pdf
✅ PDF generado: 0222DDA12BCD_certificado.pdf
✅ PDF generado: A29E75E31E7A_certificado.pdf
✅ PDF generado: 2A52CF21CDFD_certificado.pdf
✅ PDF generado: 68B40421EE60_certificado.pdf

✅ Todos los PDFs generados.
👉 Panel izquierdo → icono 📁 → clic derecho sobre cada PDF → Descargar


/tmp/ipykernel_6846/3882275292.py:26: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(210, 9, "MINISTERIO DE EDUCACION DE GUATEMALA", align="C", ln=True)
/tmp/ipykernel_6846/3882275292.py:29: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(210, 7, "Certificado Oficial de Nivel de Ingles", align="C", ln=True)
/tmp/ipykernel_6846/3882275292.py:31: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(210, 7, "Educacion Diversificada  |  Registro en Blockchain", align="C", ln=True)
/tmp/ipykernel_6846/3882275292.py:37: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(210, 9, "Se certifica que el(la) estudiante:", align="C", ln=True)
/tmp/ipyk

In [16]:
print("=" * 55)
print("   SIMULACION DE INTENTO DE FRAUDE")
print("=" * 55)

# Mostramos el dato original del certificado antes de alterarlo
print(f"\nDato original del Bloque 1:")
print(f"  {cadena.chain[1].data['nombre']} — {cadena.chain[1].data['nivel_ingles']}")

# Simulamos que alguien intenta cambiar el nivel de ingles del certificado
cadena.chain[1].data["nivel_ingles"] = "C2 - Maestria (FALSIFICADO)"
print(f"\nDato ALTERADO por el atacante:")
print(f"  {cadena.chain[1].data['nombre']} — {cadena.chain[1].data['nivel_ingles']}")

# La blockchain detecta automaticamente que algo cambio
print("\nValidando la cadena despues del intento de fraude:")
resultado = cadena.is_chain_valid()

if not resultado:
    # Si la cadena no es valida, el fraude fue detectado
    print("\n🚨 La blockchain detecto la alteracion automaticamente.")
    print("   Cualquier cambio en un certificado invalida toda la cadena.")
    print("   El fraude es IMPOSIBLE de ocultar.")

   SIMULACION DE INTENTO DE FRAUDE

Dato original del Bloque 1:
  Andrea Morales Cifuentes — B2 - Intermedio Alto

Dato ALTERADO por el atacante:
  Andrea Morales Cifuentes — C2 - Maestria (FALSIFICADO)

Validando la cadena despues del intento de fraude:
  [FALLO] Bloque 1: hash no coincide

🚨 La blockchain detecto la alteracion automaticamente.
   Cualquier cambio en un certificado invalida toda la cadena.
   El fraude es IMPOSIBLE de ocultar.
